# From one tree to an ensemble

A **decision tree** predicts by asking a sequence of yes/no questions about the features, such as "is the tenure shorter than 17 months?" and "does the customer have fibre-optic internet?". This notebook covers the first block of Session 10. Sections 1–3 explain how a tree chooses a split, Gini impurity computed by hand, how a tree grows, why deep trees overfit, and how limiting the depth or pruning controls this. Sections 4–5 combine many trees into an **ensemble**: **bagging** and **random forests** average many deep trees trained on resampled data to reduce variance, and **gradient boosting** adds small trees one after another, each correcting the errors of the current model, to reduce bias. The second lecture of the session continues with the three boosting libraries XGBoost, LightGBM and CatBoost and with interpreting the models. The examples use the IBM Telco customer churn data (7,043 customers, 26.5 % churn) introduced in Session 8.

> **Note.** The code cells in this notebook build on each other. Run them in order. The Telco data are read from GitHub, so the first Telco block needs an internet connection.

```mermaid
flowchart TD
    root["All customers<br/>26.5 % churn"] -->|"tenure ≤ 16.5"| a["New customers"]
    root -->|"tenure > 16.5"| b["Long-standing customers"]
    a -->|"no fibre optic"| a1["Leaf: stays<br/>29 % churn"]
    a -->|"fibre optic"| a2["Leaf: churns<br/>68 % churn"]
    b -->|"no fibre optic"| b1["Leaf: stays<br/>6 % churn"]
    b -->|"fibre optic"| b2["Leaf: stays<br/>28 % churn"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## 1. Splits and Gini impurity computed by hand

### Concept

A tree starts with all training rows in one **node**, the **root**. A **split** divides a node into two **child nodes** with a rule on one feature: for a numeric feature "x ≤ threshold", for a categorical feature "x in a set of categories". A node is **pure** if all its rows have the same class.

To choose a split, the tree needs a number that says how mixed a node is. The **Gini impurity** of a node with class shares p₁, …, p_K is

G = 1 − (p₁² + p₂² + … + p_K²).

It is the probability that two rows drawn at random from the node (with replacement) belong to different classes. A pure node has G = 0; a node with two classes in equal shares has G = 1 − (0.5² + 0.5²) = 0.5.

The quality of a split is the **weighted impurity** of its children, each weighted by its share of rows. The tree tries every feature and every threshold (all midpoints between sorted distinct values) and picks the split with the lowest weighted impurity, that is the largest decrease from the parent.

**Worked example.** Ten customers, four of whom churned:

| contract | month | month | month | month | month | month | year | year | year | year |
|---|---|---|---|---|---|---|---|---|---|---|
| tenure | 1 | 2 | 3 | 5 | 20 | 30 | 8 | 25 | 40 | 60 |
| churn | 1 | 1 | 1 | 0 | 1 | 0 | 0 | 0 | 0 | 0 |

- Parent node: 4 of 10 churn, G = 1 − (0.4² + 0.6²) = 1 − 0.52 = **0.48**.
- Split "contract = month": the six monthly customers have 4 churners, G = 1 − ((4/6)² + (2/6)²) = 0.444; the four yearly customers are pure, G = 0. Weighted: 0.6 · 0.444 + 0.4 · 0 = **0.267**.
- Split "tenure < 4": the three shortest-tenure customers all churned (G = 0); the other seven contain one churner, G = 1 − ((1/7)² + (6/7)²) = 0.245. Weighted: 0.3 · 0 + 0.7 · 0.245 = **0.171**.

"tenure < 4" reduces the impurity most and is the better first split.

### Why it matters

Every tree-based model of this session repeats this one operation thousands of times. Computing it once by hand shows what the algorithm optimises, and two practical consequences: trees need no scaling of the features (only the order of values matters), and they handle non-linear effects and interactions without feature engineering.

### How it works in Python

In [2]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

toy = pd.DataFrame({"contract": ["month"] * 6 + ["year"] * 4,
                    "tenure":   [1, 2, 3, 5, 20, 30, 8, 25, 40, 60],
                    "churn":    [1, 1, 1, 0, 1, 0, 0, 0, 0, 0]})


def gini(y):
    """1 - sum of squared class shares: 0 = pure node, 0.5 = 50/50 (two classes)."""
    p = np.bincount(y, minlength=2) / len(y)
    return 1 - (p ** 2).sum()


def split_gini(mask, y):
    """Weighted Gini impurity of the two children defined by a boolean mask."""
    return mask.mean() * gini(y[mask]) + (~mask).mean() * gini(y[~mask])


y_toy = toy["churn"].to_numpy()
print(round(gini(y_toy), 3))                                                   # 0.48
print(round(split_gini(toy["contract"].eq("month").to_numpy(), y_toy), 3))     # 0.267
for t in [2.5, 4, 12.5, 25]:
    print(t, round(split_gini(toy["tenure"].lt(t).to_numpy(), y_toy), 3))
# 2.5 0.3 | 4 0.171 | 12.5 0.4 | 25 0.267   -> best: tenure < 4

stump = DecisionTreeClassifier(max_depth=1).fit(toy[["tenure"]], toy["churn"])  # a tree with one split
print(export_text(stump, feature_names=["tenure"]))
# |--- tenure <= 4.00
# |   |--- class: 1
# |--- tenure >  4.00
# |   |--- class: 0
print(stump.tree_.impurity.round(3))     # [0.48  0.    0.245]: root, left child, right child

0.48
0.267
2.5 0.3
4 0.171
12.5 0.4
25 0.267
|--- tenure <= 4.00
|   |--- class: 1
|--- tenure >  4.00
|   |--- class: 0

[0.48  0.    0.245]


scikit-learn chooses the same split. It writes the threshold as the midpoint between the neighbouring values 3 and 5 (≤ 4.0), and `tree_.impurity` stores exactly the impurities we computed by hand.

### In practice

- Clinical decision rules such as the Ottawa ankle rules, which tell emergency staff when an X-ray is needed, have the form of shallow decision trees and were derived from patient data (Stiell et al., 1992).
- CART (Classification and Regression Trees; Breiman, Friedman, Olshen & Stone, 1984), the algorithm behind scikit-learn's trees, became the standard method for building interpretable rule sets from data in medicine and credit risk.

> **Warning.** Do not confuse **Gini impurity** with the Gini coefficient of income inequality or with the "Gini" of credit scoring (2 · AUC − 1). They share a name, not a formula. scikit-learn also offers `criterion="entropy"` (information gain), which usually picks very similar splits.

> **Caution.** Forgetting the weights is the classic error: a tiny pure child with one row would otherwise look like a perfect split. The weighting by child size prevents splits that isolate single rows from looking good, but only partially, which is why deep trees still overfit (Section 3).

## 2. How a tree grows and how to read it

### Concept

A tree is grown **recursively**: split the root, then apply the same procedure to each child, and so on. The **depth** of a node is the number of splits from the root to it. A node that is not split further is a **leaf**; it predicts the majority class of its training rows, and `predict_proba` returns the class shares in the leaf. Growth stops when a node is pure or when a limit is reached: `max_depth` (maximum depth), `min_samples_leaf` (minimum rows per leaf), `min_samples_split` or `max_leaf_nodes`.

The algorithm is **greedy**: each split is the best one at that moment, without looking ahead. A split that is weak now but would allow two excellent splits below it is not found. This makes trees fast to grow, but not optimal.

### Why it matters

A small tree is a readable model: each path from the root to a leaf is an if-then rule that can be explained to a manager or a customer. That is why depth-2 or depth-3 trees are used to describe segments, even when a more accurate model is used for predictions.

### How it works in Python

In [3]:
from sklearn.model_selection import train_test_split

URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)   # 11 blanks
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
cat = X.select_dtypes(exclude="number").columns.tolist()
X_oh = pd.get_dummies(X, columns=cat, drop_first=True, dtype=int)   # scikit-learn trees need numbers
X_tr, X_te, y_tr, y_te = train_test_split(X_oh, y, test_size=0.25, stratify=y, random_state=0)
print(X_oh.shape, round(y.mean(), 3))                              # (7043, 30) 0.265

small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)
print(export_text(small, feature_names=list(X_oh.columns), show_weights=True))
# |--- tenure <= 16.50
# |   |--- InternetService_Fiber optic <= 0.50
# |   |   |--- weights: [794.00, 321.00] class: 0
# |   |--- InternetService_Fiber optic >  0.50
# |   |   |--- weights: [256.00, 551.00] class: 1
# |--- tenure >  16.50
# |   |--- InternetService_Fiber optic <= 0.50
# |   |   |--- weights: [1768.00, 114.00] class: 0
# |   |--- InternetService_Fiber optic >  0.50
# |   |   |--- weights: [1062.00, 416.00] class: 0

(7043, 30) 0.265
|--- tenure <= 16.50
|   |--- InternetService_Fiber optic <= 0.50
|   |   |--- weights: [794.00, 321.00] class: 0
|   |--- InternetService_Fiber optic >  0.50
|   |   |--- weights: [256.00, 551.00] class: 1
|--- tenure >  16.50
|   |--- InternetService_Fiber optic <= 0.50
|   |   |--- weights: [1768.00, 114.00] class: 0
|   |--- InternetService_Fiber optic >  0.50
|   |   |--- weights: [1062.00, 416.00] class: 0



The `weights` are the numbers of stayers and churners in each leaf. Read the second leaf: new customers (tenure ≤ 16.5 months) with fibre-optic internet churn in 551 of 807 cases (68 %); the tree predicts churn for them. All other leaves predict "stays", although the first and fourth leaves still have churn rates of 29 % and 28 %. `sklearn.tree.plot_tree(small, feature_names=..., filled=True)` draws the same tree:

![Depth-2 decision tree on the Telco churn data, drawn with sklearn.tree.plot_tree](../figures/tree_depth2_churn.png)

### In practice

- Telecom and subscription companies use shallow trees to describe churn segments for retention campaigns ("new fibre-optic customers on monthly contracts"), because the rules translate directly into target groups.
- Telephone triage systems such as NHS Pathways, used by the NHS 111 service in England, are organised as sequences of yes/no questions, although designed by clinicians rather than learned from data.

> **Warning.** The first split is not "the most important variable" in a causal sense. Correlated features (tenure and total charges) can replace each other, and a small change in the data can change the first split. Interpret a single tree as a description of the training data, not as a causal model.

> 📖 **Read more:** [extras/03-tree-structure.ipynb](extras/03-tree-structure.ipynb) (scikit-learn gallery): reads the arrays inside a fitted tree (`children_left`, `feature`, `threshold`, `value`) and follows the decision path of one sample and of a group of samples.

## 3. Tree depth, overfitting of deep trees and pruning

### Concept

**Depth is a hyperparameter.** A shallow tree has few leaves and can only express coarse rules: it may **underfit** (high bias). A deep tree can isolate small groups of customers, eventually single rows; it **overfits** (high variance): it fits the noise of the training data, and a small change in the data produces a very different tree. Without limits, a tree grows until all leaves are pure, and its training accuracy approaches 100 %.

Two ways to control complexity:

- **Pre-pruning** (early stopping): limit the growth with `max_depth`, `min_samples_leaf` or `max_leaf_nodes`.
- **Post-pruning**: grow a full tree, then cut back branches that add little. scikit-learn implements **minimal cost-complexity pruning** (as in CART). For a value α ≥ 0 it keeps the subtree T that minimises

  R_α(T) = R(T) + α · |T|,

  where R(T) is the total impurity of the leaves (weighted by their size) and |T| the number of leaves. α is a price per leaf: α = 0 keeps the full tree, larger α gives smaller trees. `cost_complexity_pruning_path` lists the α values at which branches disappear.

Both kinds of limits are chosen by cross-validation on the training data (Session 7), never on the test set.

### Why it matters

A deep tree's training score is meaningless as an estimate of performance. The figure shows the pattern on the churn data: training accuracy rises steadily with depth, while cross-validated accuracy peaks at a depth of about 5 and then falls.

![Training and cross-validated accuracy of a decision tree on the Telco data for max_depth 1 to 20](../figures/tree_depth_vs_score.png)

### How it works in Python

In [4]:
for depth in [1, 2, 3, 5, 8, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_tr, y_tr)
    print(depth, tree.get_n_leaves(), round(tree.score(X_tr, y_tr), 3), round(tree.score(X_te, y_te), 3))
# depth leaves train test
# 1      2    0.735 0.735   both leaves predict "stays": the majority rate
# 2      4    0.79  0.79
# 3      8    0.79  0.79
# 5     32    0.802 0.78
# 8    168    0.834 0.784
# None 1054   0.997 0.72    memorises the training data

1 2 0.735 0.735
2 4 0.79 0.79
3 8 0.79 0.79
5 32 0.802 0.78
8 168 0.834 0.784
None 1054 0.997 0.72


The unlimited tree with 1,054 leaves classifies 99.7 % of the training customers correctly but only 72 % of the test customers, worse than the depth-2 tree. Now choose the limits properly, with cross-validation on the training data only:

In [5]:
from sklearn.model_selection import GridSearchCV, cross_val_score

search = GridSearchCV(DecisionTreeClassifier(random_state=0),
                      {"max_depth": range(1, 16), "min_samples_leaf": [1, 5, 20, 50]},
                      cv=5, scoring="roc_auc").fit(X_tr, y_tr)
print(search.best_params_, round(search.best_score_, 3))   # {'max_depth': 5, 'min_samples_leaf': 50} 0.828

# post-pruning: grow fully, then cut back with a price alpha per leaf
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_tr, y_tr)
print(len(path.ccp_alphas))                                # 415 candidate alphas
for alpha in [0.0, 0.0005, 0.001, 0.002, 0.005, 0.01]:
    pruned = DecisionTreeClassifier(ccp_alpha=alpha, random_state=0)
    cv_acc = cross_val_score(pruned, X_tr, y_tr, cv=5).mean()
    print(alpha, pruned.fit(X_tr, y_tr).get_n_leaves(), round(cv_acc, 3))
# alpha leaves cv_accuracy: 0.0 1054 0.727 | 0.0005 54 0.783 | 0.001 17 0.791
#                           0.002 11 0.789 | 0.005 6 0.789  | 0.01 4 0.787

{'max_depth': 5, 'min_samples_leaf': 50} 0.828
415
0.0 1054 0.727
0.0005 54 0.783


0.001 17 0.791
0.002 11 0.789
0.005 6 0.789


0.01 4 0.787


Pruning with α = 0.001 reduces the tree from 1,054 to 17 leaves and raises the cross-validated accuracy from 0.727 to 0.791. Even small α values remove most of the branches, because most of them only separate a handful of training rows. The workbook [extras/04-cost-complexity-pruning.ipynb](extras/04-cost-complexity-pruning.ipynb) (scikit-learn gallery) plots the whole path.

### In practice

- In credit scoring, regulators and model validators expect simple, stable rules; trees used directly for decisions are typically limited to a few levels and a minimum number of applicants per leaf.
- Breiman et al. (1984) introduced cost-complexity pruning with cross-validation in the CART book; it is still the default pruning method in R's `rpart` package and in scikit-learn.

> **Warning.** **Instability.** Even a well-pruned tree changes noticeably when a few training rows change. This high variance is the main weakness of single trees and the motivation for the ensembles of Sections 4 and 5, which average many trees.

> **Tip.** `min_samples_leaf` is often the most useful single limit: it guarantees that every prediction is based on at least that many customers, which also makes the class shares in the leaves (the predicted probabilities) less noisy.

A single deep tree has low bias but high variance: it fits the training data closely and changes a lot when the data change (the instability warning above). An **ensemble** combines many models into one prediction. The next two sections show the two ways of doing it. The first code cell of Section 4 repeats the Telco preparation of Section 2 (one-hot encoded features, the same stratified split), so all scores are comparable.

```mermaid
flowchart LR
    subgraph bag["Bagging / random forest: parallel"]
        d1["Bootstrap<br/>sample 1"] --> t1["Deep tree 1"]
        d2["Bootstrap<br/>sample 2"] --> t2["Deep tree 2"]
        d3["Bootstrap<br/>sample B"] --> t3["Deep tree B"]
        t1 --> avg["Average of<br/>probabilities"]
        t2 --> avg
        t3 --> avg
    end
    subgraph boost["Boosting: sequential"]
        f0["Start: constant"] --> s1["Small tree 1<br/>fits errors"]
        s1 --> s2["Small tree 2<br/>fits remaining errors"]
        s2 --> s3["... tree M"]
        s3 --> sum["Sum of all trees<br/>× learning rate"]
    end
```

## 4. Bagging and random forests

### Concept

**Bagging** (bootstrap aggregating; Breiman, 1996) trains B models, each on a **bootstrap sample**: n rows drawn *with replacement* from the n training rows. Some rows appear several times, others not at all. The ensemble predicts the average of the B predicted probabilities (or a majority vote).

Why averaging helps: if B predictions each have variance σ² and were independent, their average would have variance σ²/B. Trees trained on bootstrap samples of the same data are not independent; with pairwise correlation ρ the variance of the average is

ρ · σ² + (1 − ρ) · σ² / B.

More trees shrink the second term, but the first term remains. To reduce it, the trees must be less similar.

A **random forest** (Breiman, 2001) does exactly that: at every split, each tree may choose only among a random subset of `max_features` features (for classification often √p of the p features). A strong feature can then not dominate the top of every tree, the trees become more diverse, ρ falls, and averaging helps more.

**Out-of-bag (OOB) estimate.** A bootstrap sample leaves out about (1 − 1/n)ⁿ ≈ e⁻¹ ≈ 37 % of the rows. Each row can therefore be predicted by the trees that did not see it. The accuracy of these predictions, the **OOB score**, is a validation estimate that comes for free.

Worked example: with 10 training rows, the chance that a given row is *not* drawn in 10 draws with replacement is (9/10)¹⁰ = 0.35; with 1,000 rows it is (999/1000)¹⁰⁰⁰ = 0.368.

### Why it matters

Random forests are accurate with almost no tuning, robust to outliers and irrelevant features, and cannot overfit by adding more trees (the average just stabilises). They are a strong default and the baseline any tabular model should beat.

### How it works in Python

In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
cat = X.select_dtypes(exclude="number").columns.tolist()
X_oh = pd.get_dummies(X, columns=cat, drop_first=True, dtype=int)
X_tr, X_te, y_tr, y_te = train_test_split(X_oh, y, test_size=0.25, stratify=y, random_state=0)


def auc(model):
    """Test ROC AUC of a fitted binary classifier."""
    return round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 3)


deep = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features="sqrt",
                            oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(auc(deep), auc(bag), auc(rf))                                # 0.658 0.814 0.844
print(round(rf.oob_score_, 3), round(rf.score(X_te, y_te), 3))      # 0.802 0.798: OOB vs test accuracy

for n in [10, 50, 100, 300]:                                       # more trees: the OOB score levels off
    m = RandomForestClassifier(n_estimators=n, min_samples_leaf=5, oob_score=True,
                               random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    print(n, round(m.oob_score_, 3))
# 10 0.789 | 50 0.801 | 100 0.802 | 300 0.804

0.658 0.814 0.844
0.802 0.798
10 0.789
50 0.801
100 0.802


.venv/lib/python3.14/site-packages/sklearn/ensemble/_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


300 0.804


One deep tree reaches a test ROC AUC of 0.658. Averaging 200 such trees (bagging) raises it to 0.814; decorrelating them (random forest) to 0.844. The OOB accuracy (0.802) is close to the test accuracy (0.798). After about 50–100 trees, more trees add little. (With 10 trees scikit-learn warns that some rows have no OOB prediction: every tree happened to include them.)

### In practice

- Random forests are a standard method for land-cover classification from satellite images; Belgiu and Drăguţ (2016) review their use in remote sensing.
- In genomics, random forests are widely used for classification from gene-expression data, where there are many more features than samples.
- The Microsoft Kinect body-part recognition (Shotton et al., 2011) classified each depth-image pixel with a randomised decision forest in real time.

> **Warning.** Like every tree model, a random forest cannot extrapolate: beyond the range of the training data it predicts the value of the outermost leaves. For a trend in time (decision volumes, prices), a forest forecasts a flat line.

> 📖 **Read more:** [extras/05-ensemble-bagging.ipynb](extras/05-ensemble-bagging.ipynb) and [extras/06-ensemble-random-forest.ipynb](extras/06-ensemble-random-forest.ipynb) (INRIA MOOC) repeat bagging and random forests step by step, on synthetic data and on the Adult census data.

## 5. Gradient boosting

### Concept

**Boosting** builds the ensemble *sequentially*. For a regression target:

1. Start with a constant prediction F₀ (the mean of y).
2. Compute the **residuals** r = y − F_{m−1}(x): what the current model still gets wrong.
3. Fit a small tree h_m (depth 2–6) to the residuals.
4. Update F_m(x) = F_{m−1}(x) + η · h_m(x), with a **learning rate** η between 0 and 1.
5. Repeat M times.

For squared error, the residuals are the negative **gradient** of the loss with respect to the prediction, hence **gradient boosting** (Friedman, 2001). For classification, the model works in **log-odds**; each tree fits the gradient of the log loss, which is y − p (observed class minus predicted probability), and the probability is p = 1 / (1 + e^(−F)).

Worked example (one row, regression): y = 10, F₀ = 6, so r₁ = 4. Suppose tree 1 predicts 4 for this row; with η = 0.5, F₁ = 6 + 0.5 · 4 = 8. Now r₂ = 2; tree 2 predicts 2, F₂ = 8 + 0.5 · 2 = 9. The prediction approaches the target in shrinking steps.

The learning rate trades speed for accuracy: small η needs more trees but usually generalises better. Unlike bagging, boosting mainly reduces **bias**, and because every tree fits the remaining errors, too many trees eventually fit noise: boosting can overfit, so the number of trees must be controlled (early stopping, in the second lecture of the session).

![Gradient boosting by hand on a noisy sine curve: the fit after 1, 5, 20 and 50 trees of depth 2 with learning rate 0.3](../figures/boosting_stages.png)

### Why it matters

Gradient-boosted trees are the most accurate standard method for tabular data in many benchmarks and competitions. The loop below is the whole idea; XGBoost, LightGBM and CatBoost add regularisation, speed, and handling of missing values and categories.

### How it works in Python

In [7]:
from sklearn.tree import DecisionTreeRegressor

# gradient boosting by hand for a regression target: each small tree fits the current residuals
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 6, 200)).reshape(-1, 1)
target = np.sin(x[:, 0]) + rng.normal(0, 0.2, 200)      # noise with variance 0.04

pred = np.full(200, target.mean())                       # step 0: predict the mean
lr = 0.3                                                 # learning rate
for step in range(1, 51):
    residual = target - pred                             # what the model still gets wrong
    tree = DecisionTreeRegressor(max_depth=2).fit(x, residual)
    pred += lr * tree.predict(x)
    if step in (1, 5, 20, 50):
        print(step, round(np.mean((target - pred) ** 2), 3))
# 1 0.294 | 5 0.067 | 20 0.031 | 50 0.02

1 0.294
5 0.067
20 0.031
50 0.02


The training error falls from 0.294 after one tree to 0.020 after 50. The noise has variance 0.04, so an error below 0.04 means the model has started to fit noise. scikit-learn's implementations do the same for classification:

In [8]:
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=0).fit(X_tr, y_tr)
hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=0).fit(X_tr, y_tr)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
print(auc(gb), auc(hgb), auc(logreg))                    # 0.846 0.844 0.844

0.846 0.844 0.844


`HistGradientBoostingClassifier` bins each feature into at most 255 intervals before searching splits, the same trick as LightGBM; it is much faster on large data, handles missing values natively and supports `categorical_features` and `class_weight`. On the churn data, boosting (0.846), the random forest (0.844) and a well-specified logistic regression (0.844) are practically equal. The churn effects are few, strong and mostly monotone, so there is little for a flexible model to add. On the Berlin Airbnb prices with all listing features, many of them weak (122 amenities, reviews, availability), LightGBM prices about €3 a night more accurately than ridge regression on the same features ([04-lecture-gradient-boosting-in-practice-and-interpretation.ipynb](04-lecture-gradient-boosting-in-practice-and-interpretation.ipynb), Section 9).

### In practice

- Gradient-boosted trees are widely used for ranking web search results; LambdaMART, a boosted-tree ranker, was part of the winning entry of the 2010 Yahoo! Learning to Rank Challenge (Burges, 2010).
- Grinsztajn, Oyallon and Varoquaux (2022) compared tree ensembles and deep learning on 45 medium-sized tabular datasets and found tree-based models, including gradient boosting, ahead in most of them.
- In the M5 competition on forecasting Walmart sales, most top solutions used LightGBM, a gradient boosting library (Makridakis et al., 2022); Session 12 returns to this.

> **Warning.** Boosting needs **small** trees (depth 3–8 or a limited number of leaves) and a moderate learning rate. Deep trees with a high learning rate overfit within a few rounds. This is the opposite of random forests, which use deep trees.

> **Caution.** Trees in boosting depend on each other, so they cannot be trained in parallel; only the split search within each tree is parallelised. Training time grows linearly with the number of trees.

> 📖 **Read more:** [extras/07-hist-gradient-boosting.ipynb](extras/07-hist-gradient-boosting.ipynb) (INRIA MOOC): shows step by step, on the California housing data, that binning the features (first with `KBinsDiscretizer`, then with `HistGradientBoostingRegressor`) makes gradient boosting much faster at the same accuracy. The worked example [03-ensemble-gradient-boosting.ipynb](03-ensemble-gradient-boosting.ipynb) (INRIA MOOC) shows boosting as fitting residuals step by step.

## Check your understanding

1. A node has 30 churners and 70 stayers. What is its Gini impurity?
2. A split produces a left child with 20 rows (G = 0.1) and a right child with 80 rows (G = 0.4). What is the weighted impurity?
3. Why do decision trees not need feature scaling, while k-nearest neighbours does?
4. The unlimited tree has a training accuracy of 0.997 and a test accuracy of 0.72. Which two hyperparameters would you tune first, and with which data?
5. What happens to the number of leaves when `ccp_alpha` increases, and why?
6. What share of the training rows does a bootstrap sample of 10,000 draws from 10,000 rows leave out, approximately?
7. Why does a random forest consider only a random subset of features at each split?
8. In the worked boosting example, what would F₂ be with a learning rate of 1.0? Why is a smaller learning rate usually better?
9. Bagging reduces variance, boosting mainly reduces bias. Explain this with the depth of the trees each method uses.
10. Why can a random forest not forecast a growing trend?

## Further reading

- James, G., Witten, D., Hastie, T., Tibshirani, R. & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*, Chapter 8: Tree-Based Methods (8.1 trees; 8.2 bagging, random forests, boosting; 8.3 lab). Springer. Free PDF: https://www.statlearning.com/
- scikit-learn developers (2025). *Decision Trees*. scikit-learn user guide, section 1.10. https://scikit-learn.org/stable/modules/tree.html
- Breiman, L., Friedman, J. H., Olshen, R. A. & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.
- Amazon Machine Learning University (2021). *MLU-Explain: Decision Trees*. Interactive article. https://mlu-explain.github.io/decision-tree/
- Breiman, L. (2001). Random forests. *Machine Learning*, 45, 5–32. https://doi.org/10.1023/A:1010933404324
- Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *Annals of Statistics*, 29(5), 1189–1232. https://doi.org/10.1214/aos/1013203451
- Grinsztajn, L., Oyallon, E. & Varoquaux, G. (2022). Why do tree-based models still outperform deep learning on typical tabular data? *NeurIPS Datasets and Benchmarks*. https://arxiv.org/abs/2207.08815

> 📖 **Read more:** [extras/08-islp-tree-based-methods-lab.ipynb](extras/08-islp-tree-based-methods-lab.ipynb) (ISLP labs): the Chapter 8 lab in Python: classification and regression trees with cost-complexity pruning chosen by cross-validation, then bagging and random forests (with test error and feature importances), boosting with different learning rates and BART on the Carseats and Boston data. Needs the package `ISLP`.